In [0]:
# Import all necessary libraries
import os
from dotenv import load_dotenv
load_dotenv()

In [0]:
# Connect the storage account to the Spark session
storage_account = os.getenv("storage_account")
application_id = os.getenv("application_id")
directory_id = os.getenv("directory_id")
secret_credentials = os.getenv("secret_credentials")

required_values = {
    "storage_account": storage_account,
    "application_id": application_id,
    "directory_id": directory_id,
    "secret_credentials": secret_credentials,
}

missing = [name for name, value in required_values.items() if not value]

if missing:
    raise ValueError(
        f"Missing environment variables: {', '.join(missing)}"
    )

account_host = f"{storage_account}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{account_host}",
    "OAuth",
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{account_host}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{account_host}",
    application_id,
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{account_host}",
    secret_credentials,
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{account_host}",
    f"https://login.microsoftonline.com/{directory_id}/oauth2/token",
)

print(f"Configured OAuth for {account_host}")

In [0]:
# Read the CSV file into a Spark DataFrame
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(
        "abfss://esoolistdata@esooliststorageaccount.dfs.core.windows.net/"
        "bronze/olist_customers_dataset.csv"
    )
)

display(df)

In [0]:
base_path = (
    "abfss://esoolistdata@esooliststorageaccount.dfs.core.windows.net/"
        "bronze/"
)

orders_path = base_path + "olist_orders_dataset.csv"
payments_path = base_path + "olist_order_payments_dataset.csv"
reviews_path = base_path + "olist_order_reviews_dataset.csv"
items_path = base_path + "olist_order_items_dataset.csv"
customers_path = base_path + "olist_customers_dataset.csv"
sellers_path = base_path + "olist_sellers_dataset.csv"
geolocation_path = base_path + "olist_geolocation_dataset.csv"
products_path = base_path + "olist_products_dataset.csv"

orders_df = (
    spark.read.format("csv").option("header", "true").load(orders_path)
)
payments_df = (
    spark.read.format("csv").option("header", "true").load(payments_path)
)
reviews_df = (
    spark.read.format("csv").option("header", "true").load(reviews_path)
)
items_df = (
    spark.read.format("csv").option("header", "true").load(items_path)
)
customers_df = (
    spark.read.format("csv").option("header", "true").load(customers_path)
)
sellers_df = (
    spark.read.format("csv").option("header", "true").load(sellers_path)
)
geolocation_df = (
    spark.read.format("csv").option("header", "true").load(geolocation_path)
)
products_df = (
    spark.read.format("csv").option("header", "true").load(products_path)
)

In [0]:
display(orders_df)

In [0]:
# importing module
from pymongo import MongoClient

hostname = "5s0b6j.h.filess.io"
database = "olistmongo_darksystem"
port = "61034"
username = "olistmongo_darksystem"
password = "2ba9c3c0e9afc60ff3e1ebe78003b0e81107ea57"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]
mydatabase


In [0]:
import pandas as pd

collection = mydatabase["product_categories"]

mongo_data = pd.DataFrame(list(collection.find()))

mongo_data.head()

### Cleaning Data

In [0]:
from pyspark.sql.functions import *

In [0]:
def clean_deframe(df, name):
    print("Cleaning ", name)
    return df.dropDuplicates().na.drop('all')

orders_df = clean_deframe(orders_df, "Orders")
display(orders_df)

In [0]:
orders_df.printSchema()

In [0]:
# Convert Date Columns
orders_df = orders_df.withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp")))\
    .withColumn("order_approved_at", to_date(col("order_approved_at")))\
    .withColumn("order_delivered_carrier_date", to_date(col('order_delivered_carrier_date')))\
    .withColumn("order_delivered_customer_date", to_date(col('order_delivered_customer_date')))\
    .withColumn("order_estimated_delivery_date", to_date(col('order_estimated_delivery_date')))


In [0]:
display(orders_df)


In [0]:
from pyspark.sql.functions import col, datediff, when

# Calculate delivery times and delays
orders_df = orders_df.withColumn(
    "actual_delivery_time",
    datediff("order_delivered_customer_date", "order_purchase_timestamp"),
)

orders_df = orders_df.withColumn(
    "estimated_delivery_time",
    datediff("order_estimated_delivery_date", "order_purchase_timestamp"),
)

orders_df = orders_df.withColumn(
    "delay",
    when(
        col("actual_delivery_time") > col("estimated_delivery_time"),
        1,
    ).otherwise(0),
)

display(orders_df)

In [0]:
# Join orders with customers using customer_id
orders_customers_df = orders_df.join(
    customers_df,
    on="customer_id",
    how="left",
)

# Join payments using order_id
orders_payments_df = orders_customers_df.join(
    payments_df,
    on="order_id",
    how="left",
)

# Join order items using order_id
orders_items_df = orders_payments_df.join(
    items_df,
    on="order_id",
    how="left",
)

# Join products using product_id
orders_items_products_df = orders_items_df.join(
    products_df,
    on="product_id",
    how="left",
)

# Join sellers using seller_id
final_df = orders_items_products_df.join(
    sellers_df,
    on="seller_id",
    how="left",
)

display(final_df)

In [0]:
mongo_data.drop('_id', axis=1, inplace=True)
mongo_spark_df = spark.createDataFrame(mongo_data)
display(mongo_spark_df)

In [0]:
final_df = final_df.join(mongo_spark_df, "product_category_name", "left")

In [0]:
display(final_df)

In [0]:
def remove_duplicate_columns(df):
    columns = df.columns

    seen_columns = set()
    columns_to_drop = []

    for column in columns:
        if column in seen_columns:
            columns_to_drop.append(column)
        else:
            seen_columns.add(column)

    df_cleaned = df.drop(*columns_to_drop)
    return df_cleaned


final_df = remove_duplicate_columns(final_df)

In [0]:
# Write to Delta Lake
final_df.write.format("delta").mode("overwrite").save("abfss://esoolistdata@esooliststorageaccount.dfs.core.windows.net/"
        "silver")